In [3]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import torch
import typing as t
from torch.utils.data import TensorDataset, DataLoader

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps
from matplotlib import pyplot as plt

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


# 1. Data Loading

In [5]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('../Data/train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('../Data/kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

# 2. To dataframe

In [6]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train['full_text'] = X_train.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle['full_text'] = X_kaggle.apply(lambda tweet: extract_full_text(tweet), axis=1)

# Encoder structure

In [7]:
# encode_text = "camembert-base"
# encoder_text = "flaubert/flaubert_small_cased"
encoder_text = "cmarkea/distilcamembert-base" 


# Tweet Dataset

In [8]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(encoder_text)
max_len = 64
tokenizer.vocab_size

32005

In [9]:
from torch.utils.data import Dataset

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.float)
        }

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.05, random_state=42)

In [11]:
from torch.utils.data import DataLoader

train_dataset = TweetDataset(
    texts=X_train["full_text"].tolist(),
    labels=y_train.tolist(),
    tokenizer=tokenizer,
    max_len=max_len
)

test_dataset = TweetDataset(
    texts=X_test["full_text"].tolist(),
    labels=y_test.tolist(),
    tokenizer=tokenizer,
    max_len=max_len
)

kaggle_dataset = TweetDataset(
    texts=X_kaggle["full_text"].tolist(),
    labels=[0]*len(X_kaggle),  # Dummy labels since we don't have true labels for Kaggle test set
    tokenizer=tokenizer,
    max_len=max_len
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
test_loader_kaggle = DataLoader(kaggle_dataset, batch_size=16, shuffle=False)

In [12]:
batch = next(iter(train_loader))

ids = batch["input_ids"]
print("ids dtype:", ids.dtype)
print("ids shape:", ids.shape)
print("min id:", ids.min().item(), "max id:", ids.max().item())

print("vocab_size:", tokenizer.vocab_size)


ids dtype: torch.int64
ids shape: torch.Size([16, 64])
min id: 1 max id: 30589
vocab_size: 32005


In [13]:
from torch import nn

class CamembertClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_text)
        hidden = self.encoder.config.hidden_size     # 768
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask):
        # input shape : [batch_size, seq_len]
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ) # shape [batch_size, seq_len, hidden_size]
        cls = outputs.last_hidden_state[:, 0, :]  # vecteur [CLS]
        # cls shape : [batch_size, hidden_size]
        x = self.dropout(cls)
        logits = self.fc(x)
        # logits shape : [batch_size, num_classes]
        return logits

In [14]:
model = CamembertClassifier().to(device)

# for param in model.encoder.parameters():
#     param.requires_grad = False

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Some weights of CamembertModel were not initialized from the model checkpoint at cmarkea/distilcamembert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["label"].to(device).unsqueeze(1)
logits = model(input_ids, attention_mask)

# Training

In [16]:
def train_model(model, train_loader, test_loader, optimizer, loss_criterion, num_epochs):
    iter = 0
    history_train_acc, history_val_acc, history_train_loss, history_val_loss = [], [], [], []
    best_accuracy = 0
    for epoch in range(num_epochs):
        for i, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            # Training mode
            model.train()

            # Clear gradients with respect to parameters
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            iter += 1

            if iter % 100 == 0:
                train_loss = loss.data.item()

                model.eval()

                correct = 0
                total = 0
                with torch.no_grad():
                    for batch in test_loader:
                        input_ids = batch['input_ids'].to(device)
                        attention_mask = batch['attention_mask'].to(device)
                        labels = batch['label'].to(device).unsqueeze(1)


                        outputs = model(input_ids, attention_mask)  # (batch_size, nclasses)
                        
                        val_loss = loss_criterion(outputs, labels)

                        # multiclass prediction: choose argmax
                        probs = torch.sigmoid(outputs)
                        predicted = (probs > 0.5).float()
                        
                        total += labels.size(0)
                        correct += (predicted.cpu() == labels.cpu()).sum().item()
                
                accuracy = 100. * correct / total

                print(f'Iter: {iter:4} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss.item():2.3f} | Val Accuracy: {accuracy:.2f}')
                history_val_loss.append(val_loss.data.item())
                history_val_acc.append(round(accuracy, 2))
                history_train_loss.append(train_loss)

                # Save model when accuracy beats best accuracy
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    # We can load this best model on the validation set later
                    torch.save(model.state_dict(), 'best_model.pth')
    return (history_train_acc, history_val_acc, history_train_loss, history_val_loss)

In [17]:
train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs=4)

Iter:  100 | Train Loss: 0.581 | Val Loss: 0.288 | Val Accuracy: 60.42
Iter:  200 | Train Loss: 0.654 | Val Loss: 0.278 | Val Accuracy: 62.55
Iter:  300 | Train Loss: 0.527 | Val Loss: 0.194 | Val Accuracy: 63.75
Iter:  400 | Train Loss: 0.651 | Val Loss: 0.202 | Val Accuracy: 64.23
Iter:  500 | Train Loss: 0.580 | Val Loss: 0.244 | Val Accuracy: 65.74
Iter:  600 | Train Loss: 0.782 | Val Loss: 0.361 | Val Accuracy: 65.96
Iter:  700 | Train Loss: 0.484 | Val Loss: 0.286 | Val Accuracy: 66.96
Iter:  800 | Train Loss: 0.548 | Val Loss: 0.210 | Val Accuracy: 66.89
Iter:  900 | Train Loss: 0.422 | Val Loss: 0.190 | Val Accuracy: 67.25
Iter: 1000 | Train Loss: 0.419 | Val Loss: 0.157 | Val Accuracy: 66.81
Iter: 1100 | Train Loss: 0.605 | Val Loss: 0.208 | Val Accuracy: 67.63
Iter: 1200 | Train Loss: 0.530 | Val Loss: 0.213 | Val Accuracy: 67.29
Iter: 1300 | Train Loss: 0.570 | Val Loss: 0.276 | Val Accuracy: 66.41
Iter: 1400 | Train Loss: 0.560 | Val Loss: 0.177 | Val Accuracy: 67.07
Iter: 

KeyboardInterrupt: 

In [67]:
def save_model(model):
    model.eval()

    all_preds = []

    with torch.no_grad():
        for batch in test_loader_kaggle:  # batch from Kaggle vectorized data
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)
            logits = model(input_ids, attention_mask)                    # (batch, 1)

            probs = torch.sigmoid(logits).cpu().numpy()  # convertir en probas
            predicted = (probs > 0.5).astype(int)
            all_preds.append(predicted)

    # concatène tous les batches
    all_preds = np.vstack(all_preds).reshape(-1)

    # création du fichier Kaggle
    output = pd.DataFrame({
        "ID": X_kaggle['challenge_id'],
        "Prediction": all_preds
    })

    output.to_csv("submission.csv", index=False)
    print("Submission saved.")

In [68]:
best_model = CamembertClassifier().to(device)
best_model.load_state_dict(torch.load("best_model.pth", map_location=device))

save_model(best_model)

Submission saved.
